# Make a 2D/3D box around Aso, add Cerjean boundaries, propagate elastic and acoustic waves and thermal diffusion

In [ ]:
# Locate flexOPT securely without relying on @__DIR__ (unreliable in IJulia).
# If this notebook is outside the repository, set ENV["FLEXOPT_ROOT"] first.
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()

include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))
using .commonBatchs, .planet1D, .GeoPoints
include(joinpath(flexopt_root, "src", "seismicStations.jl"))
using .seismicStations


In [ ]:
using GLMakie
GLMakie.activate!()

# making an initial model

In [ ]:
p0=GeoPoint(dmsToDecimal(32,53,04),dmsToDecimal(131,06,14)) # Aso

Δx = 50.0 # in metre
Δy = 50.0
Δz = 50.0
altMax = 2.e3 # in metre
altMin = -10.e3 # in metre
horizontalDepth=5.e3
boxGrids3D=constructLocalBox(p0,Δx,Δy,Δz,-horizontalDepth,horizontalDepth,-horizontalDepth,horizontalDepth,altMin,altMax)

# The source may be a URL or a local ALJ-format ZIP file. Change only this
# value to use a mirror, a future NIED release, or a downloaded archive.
niedVelocitySource = DEFAULT_NIED_VELOCITY_SOURCE[]
# niedVelocitySource = "/absolute/path/to/ALJ2023.zip"

# Include the source identity in the outer model cache name. The NIED archive
# and parsed arrays also have their own cache under DEFAULT_NIED_VELOCITY_CACHE.
velocitySourceKey = nied_velocity_source_key(niedVelocitySource)
modelCacheName = "seismicModel3D_Aso_NIED_$(velocitySourceKey)"
seismicModel3D = lazyProduceOrLoad(
    modelCacheName,
    getParamsAndTopo,
    boxGrids3D.allGridsInGeoPoints,
    boxGrids3D.effectiveRadii,
    0.1;
    velocity_model=:NIED,
    nied_source=niedVelocitySource,
    nied_confidence_max=0.8,
    nied_outside=:planet1D,
    nied_low_confidence=:planet1D,
)

@show seismicModel3D.velocity_model
@show seismicModel3D.nied_source
@show count(seismicModel3D.nied_mask) / length(seismicModel3D.nied_mask)

## NIED three-dimensional velocity model

In [ ]:
# Central local x-z section. Values outside NIED coverage or above the
# confidence threshold retain the planet1D fallback.
xVelocity = [p.xyz[1] for p in boxGrids3D.allGridsInCartesian[:, 1, 1]] .* 1e-3
zVelocity = [p.xyz[3] for p in boxGrids3D.allGridsInCartesian[1, 1, :]] .* 1e-3
middleY = cld(boxGrids3D.Ny, 2)

velocityFigure = Figure(size=(1100, 450))
vpAxis = Axis(velocityFigure[1, 1]; xlabel="local x (km)", ylabel="z (km)", title="NIED/planet1D Vp")
vsAxis = Axis(velocityFigure[1, 2]; xlabel="local x (km)", ylabel="z (km)", title="NIED/planet1D Vs")
vpPlot = heatmap!(vpAxis, xVelocity, zVelocity, seismicModel3D.Vpv[:, middleY, :]; colormap=:turbo)
vsPlot = heatmap!(vsAxis, xVelocity, zVelocity, seismicModel3D.Vsv[:, middleY, :]; colormap=:turbo)
Colorbar(velocityFigure[2, 1], vpPlot; vertical=false, label="Vp (km/s)")
Colorbar(velocityFigure[2, 2], vsPlot; vertical=false, label="Vs (km/s)")
velocityFigure

In [ ]:
# Cartesian coordinates in km for plotting.
x = [p.xyz[1] for p in boxGrids3D.allGridsInCartesian[:, 1, 1]] .* 1e-3
y = [p.xyz[2] for p in boxGrids3D.allGridsInCartesian[1, :, 1]] .* 1e-3
z = [p.xyz[3] for p in boxGrids3D.allGridsInCartesian[1, 1, :]] .* 1e-3

ρ = seismicModel3D.ρ
air_density_cutoff = 0.01 # getParamsAndTopo uses ρAir = 0.001 by default
material = ρ .> air_density_cutoff

# Locate the material-to-air interface in every vertical column. The
# midpoint gives a surface with vertical uncertainty ≤ Δz/2 (50 m here).
topography = Matrix{Float64}(undef, length(x), length(y))
for j in eachindex(y), i in eachindex(x)
    k = findlast(@view material[i, j, :])
    isnothing(k) && error("No material in column (i, j) = $((i, j))")
    k == length(z) && error("Topography reaches altMax in column (i, j) = $((i, j)); increase altMax")
    all(@view material[i, j, 1:k]) || error("Material mask has a cavity at $((i, j))")
    topography[i, j] = (z[k] + z[k + 1]) / 2
end

# Upward unit normal n = (-∂h/∂x, -∂h/∂y, 1) / ‖·‖.
# Keep these arrays: they are the geometry needed by a later free-surface BC.
dh_dx = similar(topography)
dh_dy = similar(topography)
dh_dx[1, :] .= (topography[2, :] .- topography[1, :]) ./ (x[2] - x[1])
dh_dx[end, :] .= (topography[end, :] .- topography[end-1, :]) ./ (x[end] - x[end-1])
dh_dx[2:end-1, :] .= (topography[3:end, :] .- topography[1:end-2, :]) ./ (x[3:end] .- x[1:end-2])
dh_dy[:, 1] .= (topography[:, 2] .- topography[:, 1]) ./ (y[2] - y[1])
dh_dy[:, end] .= (topography[:, end] .- topography[:, end-1]) ./ (y[end] - y[end-1])
dh_dy[:, 2:end-1] .= (topography[:, 3:end] .- topography[:, 1:end-2]) ./ permutedims(y[3:end] .- y[1:end-2])

normal_scale = @. inv(sqrt(dh_dx^2 + dh_dy^2 + 1))
surface_normals = (
    nx = (@. -dh_dx * normal_scale),
    ny = (@. -dh_dy * normal_scale),
    nz = normal_scale,
)

# GMT-like shaded relief: elevation colors, GL lighting, and contours.
fig = Figure(size = (1000, 800))
ax = Axis3(
    fig[1, 1];
    xlabel = "local x (km)", ylabel = "local y (km)", zlabel = "elevation (km)",
    title = "Aso topography",
    aspect = :data,
)
terrain = surface!(
    ax, x, y, topography;
    color = topography,
    colormap = :terrain,
    colorrange = extrema(topography),
)
contour3d!(
    ax, x, y, topography;
    levels = range(extrema(topography)...; length = 15),
    color = (:black, 0.35),
    linewidth = 1,
)
Colorbar(fig[1, 2], terrain; label = "elevation (km)")

@show extrema(topography) extrema(surface_normals.nz)
fig

## Available seismic stations

In [ ]:
# Query the current public NIED station metadata, then retain stations in
# this model's geographic/local Cartesian box. Waveform download is a
# separate operation and requires a free NIED account.
stationBounds = StationBounds(boxGrids3D.allGridsInGeoPoints)
niedStations = fetch_nied_stations(bounds=stationBounds)

stationOverlay = plot_stations!(
    ax,
    niedStations,
    boxGrids3D;
    units=:km,
    show_boreholes=true,
    labels=true,
)

println("NIED stations in geographic request: ", length(niedStations))
println("NIED stations inside local box: ", length(stationOverlay.stations))
fig

In [ ]:
# Global alternative (EarthScope/IRIS-compatible FDSN service).
# Channel-level queries preserve instrument depth. Uncomment when needed.
# fdsnStations = fetch_fdsn_stations(
#     stationBounds;
#     network="*",
#     channel="HHZ,BHZ,EHZ",
# )
# fdsnOverlay = plot_stations!(ax, fdsnStations, boxGrids3D; units=:km)
# fig